# ☕ Barista Bot RAG — V1.0

Welcome to **Barista Bot**, a question-answering assistant that responds **exclusively** based on the content of a coffee book — even though that book exists only as a **scanned PDF (image)**.

### The critical data detail

Our single source of truth is a **scanned** PDF, meaning it has no searchable text layer. This means traditional libraries like `PyPDF2` or `pdfplumber` **do not work** — they read text embedded in the PDF, and here there is no text at all, only page images.

The solution is to treat each page as an **image** and apply **OCR (Optical Character Recognition)** to extract the text. This notebook implements that pipeline from scratch, with page-level traceability (`page` metadata), and builds a full RAG architecture on top of it.

### What you'll find here

0. **Colab environment** — cloning the repo, mounting Drive, and locating the PDF.
1. **Setup** — system dependencies (Tesseract OCR + Portuguese language pack) and Python libraries.
2. **Ollama** — we start a local LLM server (`llama3`) in the background, right inside Colab.
3. **OCR ingestion** — converting the PDF into images and extracting text, with page metadata.
4. **Vectorization** — chunking, embeddings, and storage in a FAISS index.
5. **RAG Chain** — a prompt that **forbids** the AI from using knowledge outside the book.
6. **Interaction** — questions and answers citing the source page.

> 🔮 This is **V1.0**: a linear, deterministic RAG pipeline. **V2.0** will evolve this same pipeline into an **agentic** architecture (see the repository README for the roadmap).

## 0. Colab environment — Clone the repository and locate the PDF

This cell sets up the environment from scratch, the way Google Colab hands you a clean machine every session:

1. **Clones this repository from GitHub** into `/content`, bringing the notebook, the README, and the `.gitignore` — but **not** the book's PDF (it's kept out of Git on purpose, since it's a data file).
2. **Mounts Google Drive**, where you should keep the scanned PDF of the book saved (avoids having to upload it manually every session).
3. **Copies the PDF from Drive** into the `data/` folder of the cloned repository, which is where the `load_documents()` function (further below) will look for it.

> ✏️ **Before running**, edit the `DRIVE_PDF_FOLDER` variable in the cell below, replacing the placeholder with the actual path of your Drive folder where the PDF is stored. This edit is local only — you don't need to (and shouldn't) commit your personal Drive path back into the public repository.

In [ ]:
import os
import shutil

REPO_URL = "https://github.com/wdasilvamf/barista-bot-rag.git"
REPO_DIR = "/content/barista-bot-rag"

# 1. Clone the repository (only if it doesn't already exist in this session)
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already cloned in this session.")

%cd {REPO_DIR}

# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# 3. Copy the PDF from Drive into the repository's data/ folder
#    <<< EDIT HERE: replace with the actual path in YOUR Google Drive >>>
DRIVE_PDF_FOLDER = "/content/drive/MyDrive/YOUR_PATH_HERE"
DATA_DEST_FOLDER = os.path.join(REPO_DIR, "data")

copied_pdfs = []
for file in os.listdir(DRIVE_PDF_FOLDER):
    if file.lower().endswith(".pdf"):
        shutil.copy(os.path.join(DRIVE_PDF_FOLDER, file), DATA_DEST_FOLDER)
        copied_pdfs.append(file)

if copied_pdfs:
    print(f"✅ PDF(s) copied to {DATA_DEST_FOLDER}: {copied_pdfs}")
else:
    print(f"⚠️ No PDF found in '{DRIVE_PDF_FOLDER}'. Check the path.")

## 1. Setup — System and Python dependencies

Since the pipeline depends on **OCR**, we need to install via `apt-get` the `tesseract-ocr` engine and the **Portuguese** language pack (`tesseract-ocr-por`) — without it, Tesseract won't correctly recognize accented characters and PT-BR words.

We also install `poppler-utils`, a system dependency required by `pdf2image` to rasterize PDF pages into images.

In [ ]:
# --- SYSTEM dependencies (via apt-get) ---
# tesseract-ocr        -> OCR engine
# tesseract-ocr-por    -> Portuguese language pack for Tesseract
# poppler-utils        -> required by pdf2image to convert PDF -> image
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por poppler-utils

# --- PYTHON dependencies ---
!pip install -q pytesseract pdf2image langchain langchain-core langchain-community \
    faiss-cpu sentence-transformers huggingface_hub pypdf

## 2. Ollama — Starting the local LLM in the background

We use [Ollama](https://ollama.com/) to run the **`llama3`** model locally inside the Colab environment, avoiding dependency on paid external APIs.

The process below:
1. Installs Ollama via the official script.
2. Starts the Ollama server (`ollama serve`) in the **background**, using `subprocess.Popen`, so the cell doesn't block waiting for it to finish.
3. Waits for the server to become available.
4. Pulls the `llama3` model.

In [ ]:
import subprocess
import time
import shutil
import requests

# 0. Install zstd, required by the Ollama install script to extract its
#    archive. Without it, install.sh fails silently on the extraction step
#    and no 'ollama' binary ends up on PATH.
!apt-get install -y -qq zstd

# 1. Install Ollama (official script)
!curl -fsSL https://ollama.com/install.sh | sh

if shutil.which("ollama") is None:
    raise RuntimeError(
        "'ollama' binary not found after installation. Check the install.sh "
        "output above for errors before continuing."
    )

# 2. Start the Ollama server in the background.
#    We use Popen (not subprocess.run) because 'ollama serve' is a
#    long-running process: run() would block waiting for it to exit.
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# 3. Wait for the server to respond before proceeding
OLLAMA_URL = "http://localhost:11434"
for attempt in range(30):
    try:
        if requests.get(OLLAMA_URL).status_code == 200:
            print("✅ Ollama server is up.")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not respond in time. Check the installation.")

# 4. Download the llama3 model (may take a few minutes the first time)
!ollama pull llama3

## 3. Data ingestion — OCR of the scanned PDF

This is the **core** step of the project. Since the PDF is a sequence of scanned pages (images), the flow is:

1. **`pdf2image.convert_from_path`** converts each PDF page into an image (`PIL.Image`).
2. **`pytesseract.image_to_string`** applies OCR to each image, using `lang="por"` to correctly recognize Portuguese.
3. Each extracted piece of text is wrapped in a LangChain `Document`, with `{"page": N}` metadata — this is **vital**: it's what lets the end user know exactly which page of the book each answer came from.
4. The full text is cached (`data/ocr_cache.json`) so we don't need to reprocess the entire PDF (a slow process) every time the notebook runs.

In [ ]:
import os
import json
from pdf2image import convert_from_path
import pytesseract
from langchain_core.documents import Document

# REPO_DIR is defined by the Colab setup cell above (section 0).
# When running there, the PDF was copied into REPO_DIR/data. When running
# outside Colab (no REPO_DIR defined), fall back to a local ./data folder.
DATA_DIR = os.path.join(REPO_DIR, "data") if "REPO_DIR" in globals() else "data"
CACHE_PATH = os.path.join(DATA_DIR, "ocr_cache.json")


def find_pdf(data_dir: str) -> str:
    """Locates the first PDF inside the data folder."""
    pdfs = [f for f in os.listdir(data_dir) if f.lower().endswith(".pdf")]
    if not pdfs:
        raise FileNotFoundError(
            f"No PDF found in '{data_dir}'. Upload the scanned book to that folder."
        )
    return os.path.join(data_dir, pdfs[0])


def extract_text_via_ocr(pdf_path: str, language: str = "por", dpi: int = 300) -> list[Document]:
    """Converts each page of the scanned PDF into an image and applies OCR.

    Returns a list of LangChain Documents, one per page, each with
    metadata {'page': page_number, 'source': file_name}. This page
    metadata is what later allows telling the user exactly where in
    the book each answer came from.
    """
    print(f"📄 Converting pages of '{pdf_path}' into images (dpi={dpi})...")
    page_images = convert_from_path(pdf_path, dpi=dpi)

    documents = []
    for page_number, image in enumerate(page_images, start=1):
        print(f"🔎 OCR on page {page_number}/{len(page_images)}...", end="\r")
        text = pytesseract.image_to_string(image, lang=language)
        text = text.strip()

        if not text:
            # Page with no recognizable text (e.g. blank cover) -> skip
            continue

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "page": page_number,
                    "source": os.path.basename(pdf_path),
                },
            )
        )
    print(f"\n✅ OCR complete: {len(documents)} pages with extracted text.")
    return documents


def load_documents(data_dir: str = DATA_DIR, force_reprocessing: bool = False) -> list[Document]:
    """Loads documents from cache if it exists; otherwise runs OCR."""
    if os.path.exists(CACHE_PATH) and not force_reprocessing:
        print("⚡ OCR cache found. Loading already extracted text...")
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            raw = json.load(f)
        return [Document(page_content=d["page_content"], metadata=d["metadata"]) for d in raw]

    pdf_path = find_pdf(data_dir)
    documents = extract_text_via_ocr(pdf_path)

    # Save cache to speed up future runs
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(
            [{"page_content": d.page_content, "metadata": d.metadata} for d in documents],
            f,
            ensure_ascii=False,
        )
    return documents


documents = load_documents()
print(f"\nExample (page {documents[0].metadata['page']}):\n")
print(documents[0].page_content[:500])

## 4. Vectorization — Chunking, Embeddings, and FAISS

With the text extracted (and the source page preserved in each `Document`), we follow the standard RAG pipeline:

1. **`RecursiveCharacterTextSplitter`** splits each page's text into smaller, more cohesive pieces (*chunks*), preserving the source `page` metadata in every generated chunk.
2. **Embeddings** — we use HuggingFace's `BAAI/bge-small-en-v1.5` model, lightweight and efficient enough to run on CPU.
3. **FAISS** — we store the vectors in a local FAISS index, enabling semantic similarity search.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Chunking: splits long pages into smaller, more coherent pieces
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(documents)
print(f"📚 {len(documents)} pages split into {len(chunks)} chunks.")

# 2. Embeddings: lightweight model, runs well on CPU
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

# 3. FAISS vector index
vector_store = FAISS.from_documents(chunks, embeddings)
print("✅ FAISS index created successfully.")

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

## 5. RAG Chain — Restrictive prompt + Ollama (llama3)

Here we assemble the chain that connects the `retriever` (FAISS) to the LLM (`llama3` via Ollama).

The central point of this step is the **prompt template**: it explicitly instructs the model to **answer only based on the context retrieved from the book**, forbidding the use of external knowledge. This drastically reduces the risk of hallucination and ensures the Barista Bot stays faithful to the source.

In [ ]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

llm = Ollama(model="llama3", base_url=OLLAMA_URL, temperature=0)

# Restrictive prompt: the AI can only answer based on the provided CONTEXT.
# This prevents the model from "filling in" answers with knowledge external
# to the book, which is our single source of truth.
PROMPT_TEMPLATE = """You are the Barista Bot, a coffee expert who answers
EXCLUSIVELY based on the CONTEXT below, extracted from a book about coffee.

Strict rules:
- Do NOT use any knowledge outside the CONTEXT, even if you happen to know it.
- If the answer is not in the CONTEXT, clearly say:
  "I couldn't find that information in the book."
- Answer in English, clearly and objectively.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER (based only on the CONTEXT):"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)
print("✅ RAG chain ready.")

## 6. Interaction — Ask the Barista Bot

The function below runs the chain and prints, besides the answer, the **book pages** that grounded the answer — fulfilling the source traceability requirement.

In [ ]:
def ask(question: str):
    result = rag_chain.invoke({"query": question})
    answer = result["result"]
    sources = sorted({doc.metadata.get("page") for doc in result["source_documents"]})

    print("🤖 Answer:\n")
    print(answer)
    print("\n📖 Source(s): page(s)", ", ".join(str(p) for p in sources if p is not None))
    return result


# Example usage:
_ = ask("What is the ideal water temperature for brewing coffee?")

---
### ✅ End of V1.0

You just built a complete RAG pipeline, from the scanned PDF to a page-traceable answer, using **only** open-source tools running locally (Tesseract, FAISS, Ollama).

Next step for the project: evolve this linear pipeline into an **agentic architecture (V2.0)** — see the repository's `README.md` for the detailed roadmap.